# Lint your OpenLLMetry / Traceloop agent traces with tracelint

If you instrument your agents with **OpenLLMetry / Traceloop**, they emit **OpenTelemetry GenAI** spans. [**tracelint**](https://github.com/AshwinUgale/tracelint) runs *on top of* them: it reads the spans you already export and reports **structural** defects — ignored tool errors, schema-violating calls, hallucinated arguments, loops, **duplicate side effects** — each with the exact span as evidence and a CI exit code.

**No second model judges the trace.** For this class of bug a judge is the wrong tool, because the defect is *decidable by looking at the trace*. Deterministic, reproducible, and ~free.

## Install

```bash
pip install tracelint
```

## On the spans you already export

However you export your OpenLLMetry/Traceloop OTel spans (an OTLP JSON dump from your collector or backend), tracelint's event-list reader understands the GenAI semantic convention (`gen_ai.operation.name`) directly:

In [ ]:
from tracelint import ToolRegistry, load_source, default_rules, lint_trace, render_report

registry = ToolRegistry.load("tools.json")   # optional: side_effecting / failure_when
for trace in load_source("spans.json", "otel"):
    print(render_report(lint_trace(trace, default_rules(), registry), include_candidates=True))

At the command line, with a CI exit code (2 on a hard defect):

```bash
tracelint check spans.json --format otel --tools tools.json
```

## Run it now — offline, no collector or API key

This is an **illustrative** OTel GenAI trace (constructed, not a captured run) — swap in your own `spans.json` export from the cell above to lint real traces. Here the support agent is asked to refund order `A100`, the `get_order` lookup **errors**, and the agent refunds the card anyway — using the errored order id — and refunds it **twice**.

In [ ]:
import json
from tracelint import ToolRegistry, lint_otel_trace, render_report

TRACE_ID = "support-run"


def tool_span(sid, start, name, args, output, errored=False):
    s = {"span_id": sid, "trace_id": TRACE_ID, "start_time": start, "name": name,
         "attributes": {"gen_ai.operation.name": "execute_tool", "gen_ai.tool.name": name,
                        "input": json.dumps(args), "output": json.dumps(output)}}
    if errored:
        s["status"] = {"status_code": "ERROR", "description": "500 internal error"}
    else:
        s["status_code"] = "OK"
    return s


spans = [
    # The opening `chat` span carries gen_ai.input.messages — what the model was asked — so
    # provenance (R3) sees the order id came from the user and does not flag it.
    {"span_id": "s0", "trace_id": TRACE_ID, "start_time": "0", "status_code": "OK",
     "attributes": {"gen_ai.operation.name": "chat",
                    "gen_ai.input.messages": json.dumps(
                        [{"role": "user", "parts": [{"type": "text",
                          "content": "Refund order A100."}]}])}},
    tool_span("s1", "1", "get_order", {"order_id": "A100"},
              {"order_id": "A100", "status": "error"}, errored=True),
    tool_span("s2", "2", "refund_order", {"order_id": "A100"}, {"refunded": True}),
    tool_span("s3", "3", "refund_order", {"order_id": "A100"}, {"refunded": True}),
]

# The operator's tools.json: refund_order mutates the world, and {"refunded": false} is its
# declared failure. Declared once, never guessed from the tool name.
registry = ToolRegistry.from_dict({"tools": {
    "get_order": {},
    "refund_order": {"metadata": {"side_effecting": True,
                                  "failure_when": {"pointer": "/refunded", "equals": False}}}}})

report = lint_otel_trace(spans, registry=registry)
print(render_report(report, include_candidates=True))
print("\nexit code:", report.exit_code, " (2 = a hard defect; fails CI)")

You'll see three findings, each pointing at the exact spans:

- **R2a `hard_event`** — `get_order` returned an error (read from the OTel `status`).
- **R2b `hard_defect`** — the errored order id was **reused** as the argument to a side-effecting `refund_order`. Data from a failed call fed into a real-world action, no fallback. This is the tier that **fails CI** (exit `2`).
- **R8 `hard_event`** — `refund_order` was called **twice** with the same arguments after the first succeeded: a **double refund**.

It also discloses what it *couldn't* check (no schema for these tools → R1 suppressed) plus per-rule **verification coverage**, so a clean report is honestly clean — not just empty.

## Repo & docs

**https://github.com/AshwinUgale/tracelint** — MIT, deterministic, judge-free. Reads OpenInference/Phoenix, Langfuse, LangSmith, and OTel GenAI traces.